<a href="https://colab.research.google.com/github/anhndt0310-jpg/vietnamese-cyberbullying-detection/blob/main/vietnamese_cyberbullying_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers torch pandas numpy scikit-learn

In [2]:
import pandas as pd

# Đường dẫn chính xác đã xác nhận
train_path = '/content/drive/MyDrive/df_train_clean.csv'
test_path = '/content/drive/MyDrive/df_test_clean.csv'

# Nạp dữ liệu
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f"✅ Đã tải dữ liệu thành công!")
print(f"- Tập Train: {len(train_df)} dòng")
print(f"- Tập Test: {len(test_df)} dòng")

display(train_df.head(3))

✅ Đã tải dữ liệu thành công!
- Tập Train: 7000 dòng
- Tập Test: 1000 dòng


,Unnamed: 0,content,Constructiveness,Toxicity,Title,Topic,content_clean
0,6326,Thật tuyệt vời...!!!,0,0,Những 'bước tiến diệu kỳ' của Trúc Nhi - Diệu Nhi,SucKhoe,Thật tuyệt vời...!!!
1,7835,"mỹ đã tuột dốc quá nhiều rồi, giờ muốn vực dậy...",1,0,Hình tượng Mỹ sụp đổ trong lòng người dân thế ...,TheGioi,"mỹ đã tuột dốc quá nhiều rồi, giờ muốn vực dậy..."
2,4690,tôi thấy người lái xe hơi bấm còi mới là người...,1,1,Cả trăm người đạp xe thể dục bịt kín đường,OtoXemay,tôi thấy người lái xe hơi bấm còi mới là người...


In [3]:
import os
import numpy as np

# Đường dẫn lưu tệp trên Drive
emb_train_path = '/content/drive/MyDrive/X_train_emb.npy'
emb_test_path = '/content/drive/MyDrive/X_test_emb.npy'

# Kiểm tra nếu tệp đã tồn tại bằng os.path.exists
if os.path.exists(emb_train_path) and os.path.exists(emb_test_path):
    print("🔄 Đang nạp đặc trưng từ Google Drive... ")
    X_train_emb = np.load(emb_train_path)
    X_test_emb = np.load(emb_test_path)
    y_train = train_df['Toxicity'].values
    y_test = test_df['Toxicity'].values
    print("✅ Đã nạp xong!")
else:
    print("⚠️ Không tìm thấy file lưu sẵn. Đang trích xuất... ")
    X_train_emb = get_phobert_embeddings(train_df['content_clean'].fillna('').tolist())
    y_train = train_df['Toxicity'].values

    X_test_emb = get_phobert_embeddings(test_df['content_clean'].fillna('').tolist())
    y_test = test_df['Toxicity'].values

    # Lưu sẵn file nếu hết GPU
    np.save(emb_train_path, X_train_emb)
    np.save(emb_test_path, X_test_emb)
    print("✅ Đã trích xuất và lưu vào Drive!")

print(f"Kích thước tập Train: {X_train_emb.shape}")
print(f"Kích thước tập Test: {X_test_emb.shape}")

🔄 Đang nạp đặc trưng từ Google Drive (Bỏ qua bước trích xuất)... 
✅ Đã nạp xong!
Kích thước tập Train: (7000, 768)
Kích thước tập Test: (1000, 768)


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import joblib

# Khởi tạo và huấn luyện lại với tên biến riêng biệt
lr_final = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)

print("Đang huấn luyện Logistic Regression...")
lr_final.fit(X_train_emb, y_train)

# Lưu mô hình này vào Drive để đảm bảo file pkl là mô hình ML, không phải Transformer
joblib.dump(lr_final, '/content/drive/MyDrive/phobert_logreg_model.pkl')

# Dự đoán trên tập Test
y_pred_lr = lr_final.predict(X_test_emb)

print("\n=== KẾT QUẢ LOGISTIC REGRESSION ===")
print(classification_report(y_test, y_pred_lr))

Đang huấn luyện Logistic Regression...

=== KẾT QUẢ LOGISTIC REGRESSION ===
              precision    recall  f1-score   support

           0       0.94      0.81      0.87       890
           1       0.28      0.61      0.39       110

    accuracy                           0.79      1000
   macro avg       0.61      0.71      0.63      1000
weighted avg       0.87      0.79      0.82      1000



Thử các classifier khác

In [5]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# Random forest
rf_model = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
rf_model.fit(X_train_emb, y_train)
y_pred_rf = rf_model.predict(X_test_emb)

# XGBoost
xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    eval_metric='logloss',
    random_state=42
)
xgb_model.fit(X_train_emb, y_train)
y_pred_xgb = xgb_model.predict(X_test_emb)

print("✅ Đã huấn luyện xong Random Forest và XGBoost!")

✅ Đã huấn luyện xong Random Forest và XGBoost!


Fine-tune PhoBERT

In [8]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, AutoTokenizer
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score

# Khởi tạo tokenizer
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")

# Chuyển sang format HuggingFace Dataset
train_dataset = Dataset.from_pandas(train_df[['content_clean', 'Toxicity']].rename(columns={'content_clean': 'text', 'Toxicity': 'label'}))
test_dataset = Dataset.from_pandas(test_df[['content_clean', 'Toxicity']].rename(columns={'content_clean': 'text', 'Toxicity': 'label'}))

# Tokenize
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Load model với classification head
model = AutoModelForSequenceClassification.from_pretrained("vinai/phobert-base", num_labels=2)

# Cấu hình training
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
)

# Metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='weighted')
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transf

Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [9]:
# 1. Lưu mô hình sau khi Fine-tuning
model_save_path = '/content/drive/MyDrive/phobert_finetuned_toxicity'
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"✅ Đã lưu mô hình fine-tuned tại: {model_save_path}")

# 2. Đánh giá chi tiết trên tập test
evaluations = trainer.evaluate()
print("\n=== KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST ===")
for key, value in evaluations.items():
    print(f"{key}: {value}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Đã lưu mô hình fine-tuned tại: /content/drive/MyDrive/phobert_finetuned_toxicity


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

2. Thêm Các Metric Đánh giá Chi tiết

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, precision_recall_curve, average_precision_score,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
import numpy as np

def evaluate_model(y_true, y_pred, y_proba=None, model_name="Model"):
    print(f"\n{'='*50}")
    print(f"KẾT QUẢ ĐÁNH GIÁ: {model_name}")
    print('='*50)

    # Classification Report
    print("\n📊 Classification Report:")
    print(classification_report(y_true, y_pred, target_names=['Non-Toxic', 'Toxic']))

    # Confusion Matrix
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=['Non-Toxic', 'Toxic'], ax=ax, cmap='Blues')
    plt.title(f'Confusion Matrix - {model_name}')
    plt.tight_layout()
    plt.show()

    # ROC-AUC nếu có probability
    if y_proba is not None:
        auc = roc_auc_score(y_true, y_proba)
        ap = average_precision_score(y_true, y_proba)
        print(f"\n🎯 ROC-AUC Score: {auc:.4f}")
        print(f"🎯 Average Precision: {ap:.4f}")

# Sử dụng các biến dự đoán đã có sẵn từ cell JgyecsOHMYWz
try:
    # Nếu logreg_model chưa được định nghĩa, ta dùng biến 'model' từ cell huấn luyện Logistic
    y_proba_logreg = None
    if 'model' in globals() and hasattr(model, 'predict_proba'):
        y_proba_logreg = model.predict_proba(X_test_emb)[:, 1]

    evaluate_model(y_test, y_pred, y_proba_logreg, "PhoBERT + Logistic Regression")
except Exception as e:
    print(f"Có lỗi xảy ra khi đánh giá: {e}")
    print("Vui lòng đảm bảo bạn đã chạy cell huấn luyện Logistic Regression trước đó.")

3. Cross-Validation để đánh giá ổn định

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Cross-validation scores
cv_scores = cross_val_score(
    LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42),
    X_train_emb, y_train,
    cv=cv, scoring='f1_weighted', n_jobs=-1
)

print(f"Cross-Validation F1 Scores: {cv_scores}")
print(f"Mean F1: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")


4. Lưu và Tải Model

In [ ]:
import joblib

# Lưu model
joblib.dump(model, '/content/drive/MyDrive/phobert_logreg_model.pkl')

# Lưu embeddings (để không phải chạy lại)
np.save('/content/drive/MyDrive/X_train_emb.npy', X_train_emb)
np.save('/content/drive/MyDrive/X_test_emb.npy', X_test_emb)

# Tải lại
model = joblib.load('/content/drive/MyDrive/phobert_logreg_model.pkl')
X_train_emb = np.load('/content/drive/MyDrive/X_train_emb.npy')


5. Hàm Dự đoán cho câu mới

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
import joblib
import os

# 1. Sử dụng lại các biến đã có để tránh tải lại model
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Kiểm tra nếu chưa có model/tokenizer trong bộ nhớ thì mới load
if 'phobert' not in globals():
    print("📦 Đang nạp PhoBERT base model (lần đầu)...")
    phobert_base = AutoModel.from_pretrained("vinai/phobert-base").to(device)
else:
    phobert_base = phobert # Dùng lại biến phobert từ cell 14b8be58

if 'tokenizer' not in globals():
    tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")

# 2. Tải mô hình Logistic Regression đã huấn luyện
model_path = '/content/drive/MyDrive/phobert_logreg_model.pkl'
if os.path.exists(model_path):
    final_ml_model = joblib.load(model_path)
else:
    final_ml_model = lr_final # Dùng trực tiếp nếu file chưa kịp lưu

def predict_toxicity_ml(text, ml_model, tokenizer, embed_model):
    embed_model.eval()
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)

    with torch.no_grad():
        outputs = embed_model(**inputs)

    # Lấy vector CLS
    embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy()

    # Dự đoán
    prediction = ml_model.predict(embedding)[0]
    probability = ml_model.predict_proba(embedding)[0]

    result = "🚨 ĐỘC HẠI" if prediction == 1 else "✅ KHÔNG ĐỘC HẠI"
    print(f"Văn bản: {text}")
    print(f"Kết quả: {result} ({probability[1]:.2%} toxic)")
    print("-" * 30)

print("--- THỬ NGHIỆM DỰ ĐOÁN NHANH ---")
predict_toxicity_ml("Bạn thật tuyệt vời!", final_ml_model, tokenizer, phobert_base)
predict_toxicity_ml("Đồ ngu ngốc vô dụng!", final_ml_model, tokenizer, phobert_base)

6. So sánh nhiều Model (Tổng hợp)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1),
    'SVM': SVC(class_weight='balanced', probability=True, random_state=42),
    'MLP Neural Network': MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=500, random_state=42),
}

results = []
for name, clf in models.items():
    print(f"\n🔄 Đang huấn luyện {name}...")
    clf.fit(X_train_emb, y_train)
    y_pred = clf.predict(X_test_emb)

    from sklearn.metrics import f1_score, accuracy_score
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'F1 (Weighted)': f1_score(y_test, y_pred, average='weighted'),
        'F1 (Toxic)': f1_score(y_test, y_pred, pos_label=1)
    })

results_df = pd.DataFrame(results).sort_values('F1 (Weighted)', ascending=False)
print("\n" + "="*60)
print("📊 BẢNG SO SÁNH CÁC MÔ HÌNH")
print("="*60)
print(results_df.to_string(index=False))


In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, f1_score
from sklearn.ensemble import RandomForestClassifier
from tqdm import tqdm

# 1. Đảm bảo các thành phần PhoBERT đã sẵn sàng
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")
phobert = AutoModel.from_pretrained("vinai/phobert-base").to(device)

def get_phobert_embeddings(texts, batch_size=16):
    phobert.eval()
    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i+batch_size]
        inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
        with torch.no_grad():
            outputs = phobert(**inputs)
        # Lấy vector [CLS]
        batch_emb = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        embeddings.append(batch_emb)
    return np.vstack(embeddings)

# 2. Kiểm tra và trích xuất lại đặc trưng
if 'X_train_emb' not in globals():
    print("🔄 Đang trích xuất đặc trưng (có thể mất ít phút)...")
    X_train_emb = get_phobert_embeddings(train_df['content_clean'].fillna('').tolist())
    X_test_emb = get_phobert_embeddings(test_df['content_clean'].fillna('').tolist())
    y_train = train_df['Toxicity'].values
    y_test = test_df['Toxicity'].values

print("--- TỐI ƯU RANDOM FOREST (THRESHOLD MOVING) ---")

# 3. Huấn luyện với trọng số cực cao cho lớp Toxic
rf_optimized = RandomForestClassifier(n_estimators=300,
                                      class_weight={0: 1, 1: 12},
                                      max_depth=20,
                                      random_state=42,
                                      n_jobs=-1)

print("Đang huấn luyện Random Forest...")
rf_optimized.fit(X_train_emb, y_train)

# 4. Sử dụng Threshold Moving (Hạ xuống 0.25 để ưu tiên Recall)
y_probs_rf = rf_optimized.predict_proba(X_test_emb)[:, 1]
threshold = 0.25
y_pred_rf_custom = (y_probs_rf >= threshold).astype(int)

print(f"\nKết quả Random Forest sau khi chỉnh ngưỡng {threshold}:")
print(classification_report(y_test, y_pred_rf_custom, target_names=['Non-Toxic', 'Toxic']))

f1_toxic = f1_score(y_test, y_pred_rf_custom)
print(f"=> F1-score mới cho lớp Toxic: {f1_toxic:.4f}")